In [ ]:
import os
import sys

import pandas as pd
import numpy as np
from scipy import sparse

import seaborn as sns
from matplotlib import pyplot as plt

from qmplot import qqplot

import muon as mu

sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from Stage2_Evaluation.A_Metrics.src import (
    compute_perturbation_association,
)

In [ ]:
# ── IO paths ──
project_root = '/oak/stanford/groups/engreitz/Users/ymo/Project/IGVF_Huangfu_WTC11'
out_dir = f'{project_root}/Result'
run_name = '030726_20iter_5KHVG_torch_halsvar_batch_e7'
eval_dir_name = 'Eval'

# ── Keys ──
data_key = 'rna'
prog_key = 'cNMF'
categorical_key = 'WTC'
samples = ['WTC']

# ── Run variables ──
components = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100, 150, 200]
sel_threshs = [0.2, 2.0]

# ── Fake test variables ──
number_run = 300
number_guide = 6

# Compute real perturbation test

Real perturbation results already exist from the evaluation pipeline. Re-run only if needed.

In [ ]:
# Skip if real perturbation results already exist; set to True to force re-run
rerun_real = False

if rerun_real:
    for sel_thresh in sel_threshs:
        for k in components:
            thresh_str = str(sel_thresh).replace('.', '_')
            output_folder = f'{out_dir}/{run_name}/{eval_dir_name}/{k}_{thresh_str}'
            os.makedirs(output_folder, exist_ok=True)

            mdata_path = f'{out_dir}/{run_name}/adata/cNMF_{k}_{thresh_str}.h5mu'
            if not os.path.exists(mdata_path):
                print(f'Skipping K={k}, thresh={sel_thresh}: {mdata_path} not found')
                continue

            print(f'Processing K={k}, sel_thresh={sel_thresh}')
            mdata = mu.read(mdata_path)

            for samp in mdata[data_key].obs[categorical_key].unique():
                mdata_ = mdata[mdata[data_key].obs[categorical_key] == samp]
                test_stats_df = compute_perturbation_association(
                    mdata_, prog_key=prog_key,
                    collapse_targets=True,
                    pseudobulk=False,
                    reference_targets=['non-targeting'],
                    FDR_method='StoreyQ',
                    n_jobs=-1, inplace=False
                )
                test_stats_df.to_csv(
                    f'{output_folder}/{k}_perturbation_association_results_{samp}.txt',
                    sep='\t', index=False
                )
else:
    print('Skipping real perturbation test (results already exist). Set rerun_real=True to re-run.')

# Compute fake perturbation test (calibration)

In [ ]:
test_stats_fake_dfs = []

for sel_thresh in sel_threshs:
    for k in components:
        thresh_str = str(sel_thresh).replace('.', '_')
        output_folder = f'{out_dir}/{run_name}/{eval_dir_name}/{k}_{thresh_str}'
        os.makedirs(output_folder, exist_ok=True)

        mdata_path = f'{out_dir}/{run_name}/adata/cNMF_{k}_{thresh_str}.h5mu'
        if not os.path.exists(mdata_path):
            print(f'Skipping K={k}, thresh={sel_thresh}: {mdata_path} not found')
            continue

        print(f'Processing K={k}, sel_thresh={sel_thresh}')
        mdata = mu.read(mdata_path)

        # Identify non-targeting guides from guide_targets
        guide_targets = mdata[prog_key].uns['guide_targets']
        guide_names = mdata[prog_key].uns['guide_names']
        non_targeting_idx = np.where(guide_targets == 'non-targeting')[0]
        print(f'  Non-targeting guides: {len(non_targeting_idx)}')

        test_stats_fake_dfs_temp = []
        for i in range(number_run):
            if (i + 1) % 50 == 0:
                print(f'  Iteration {i+1}/{number_run}')

            # Randomly relabel number_guide non-targeting guides as 'targeting'
            new_targets = np.full(len(non_targeting_idx), 'non-targeting', dtype=object)
            selected = np.random.choice(len(non_targeting_idx), number_guide, replace=False)
            new_targets[selected] = 'targeting'

            # Subset mdata to only non-targeting guides with relabeled targets
            _mdata = mdata.copy()
            ga = mdata[prog_key].obsm['guide_assignment']
            if sparse.issparse(ga):
                _mdata[prog_key].obsm['guide_assignment'] = ga[:, non_targeting_idx]
            else:
                _mdata[prog_key].obsm['guide_assignment'] = ga[:, non_targeting_idx]
            _mdata[prog_key].uns['guide_names'] = guide_names[non_targeting_idx]
            _mdata[prog_key].uns['guide_targets'] = new_targets

            # Run perturbation association per sample
            for samp in _mdata[data_key].obs[categorical_key].unique():
                mdata_samp = _mdata[_mdata[data_key].obs[categorical_key] == samp]
                test_stats_df = compute_perturbation_association(
                    mdata_samp, prog_key=prog_key,
                    collapse_targets=True,
                    pseudobulk=False,
                    reference_targets=['non-targeting'],
                    FDR_method='BH',
                    n_jobs=-1, inplace=False
                )
                test_stats_df[categorical_key] = samp
                test_stats_df['K'] = k
                test_stats_df['run'] = i
                test_stats_df['sel_thresh'] = sel_thresh
                test_stats_fake_dfs.append(test_stats_df)
                test_stats_fake_dfs_temp.append(test_stats_df)

        # Save per K/threshold
        df_temp = pd.concat(test_stats_fake_dfs_temp, ignore_index=True)
        df_temp.to_csv(
            f'{output_folder}/{k}_fake_perturbation_association_calibration.txt',
            sep='\t', index=False
        )
        print(f'  Saved {output_folder}/{k}_fake_perturbation_association_calibration.txt')

In [ ]:
# Save combined fake test results
if test_stats_fake_dfs:
    test_stats_fake_all = pd.concat(test_stats_fake_dfs, ignore_index=True)
    combined_path = f'{out_dir}/{run_name}/{eval_dir_name}/perturbation_association_calibration.txt'
    test_stats_fake_all.to_csv(combined_path, sep='\t', index=False)
    print(f'Saved combined results to {combined_path}')

# Compare real vs fake (QQ plots)

In [ ]:
# Load real perturbation results
test_stats_real_df = []
for sel_thresh in sel_threshs:
    for k in components:
        thresh_str = str(sel_thresh).replace('.', '_')
        for samp in samples:
            fpath = f'{out_dir}/{run_name}/{eval_dir_name}/{k}_{thresh_str}/{k}_perturbation_association_results_{samp}.txt'
            if not os.path.exists(fpath):
                continue
            df_ = pd.read_csv(fpath, sep='\t')
            df_['sample'] = samp
            df_['K'] = k
            df_['sel_thresh'] = sel_thresh
            test_stats_real_df.append(df_)

test_stats_real_df = pd.concat(test_stats_real_df, ignore_index=True)
test_stats_real_df['real'] = True

# Load fake perturbation results
test_stats_fake_df = []
for sel_thresh in sel_threshs:
    for k in components:
        thresh_str = str(sel_thresh).replace('.', '_')
        fpath = f'{out_dir}/{run_name}/{eval_dir_name}/{k}_{thresh_str}/{k}_fake_perturbation_association_calibration.txt'
        if not os.path.exists(fpath):
            continue
        df_ = pd.read_csv(fpath, sep='\t')
        test_stats_fake_df.append(df_)

test_stats_fake_df = pd.concat(test_stats_fake_df, ignore_index=True)
test_stats_fake_df['real'] = False

# Merge
test_stats_dfs = pd.concat([test_stats_real_df, test_stats_fake_df], ignore_index=True)
print(f'Real: {len(test_stats_real_df)}, Fake: {len(test_stats_fake_df)}, Total: {len(test_stats_dfs)}')

In [ ]:
# QQ plot — real
plot_dir = f'{out_dir}/{run_name}/Plot/QQ_plot'
os.makedirs(plot_dir, exist_ok=True)

nrows = len(components) // 4 + 1
fig, axs = plt.subplots(ncols=4, nrows=nrows, figsize=(20, 5 * nrows))
for i, k in enumerate(components):
    mask = (test_stats_dfs.K == k) & (test_stats_dfs.real == True)
    if mask.sum() > 0:
        qqplot(data=test_stats_dfs.loc[mask, 'pval'], ax=axs.flat[i])
    axs.flat[i].set_title(f'K={k}')

for j in range(i + 1, len(axs.flat)):
    axs.flat[j].axis('off')
plt.tight_layout()
plt.savefig(f'{plot_dir}/perturbation_association_qqplot_real.png', dpi=100)
plt.show()

In [ ]:
# QQ plot — fake
nrows = len(components) // 4 + 1
fig, axs = plt.subplots(ncols=4, nrows=nrows, figsize=(20, 5 * nrows))
for i, k in enumerate(components):
    mask = (test_stats_dfs.K == k) & (test_stats_dfs.real == False) & (test_stats_dfs.target_name == 'targeting')
    if mask.sum() > 0:
        qqplot(data=test_stats_dfs.loc[mask, 'pval'], ax=axs.flat[i])
    axs.flat[i].set_title(f'K={k}')

for j in range(i + 1, len(axs.flat)):
    axs.flat[j].axis('off')
plt.tight_layout()
plt.savefig(f'{plot_dir}/perturbation_association_qqplot_fake.png', dpi=100)
plt.show()

In [ ]:
# QQ plot — real vs fake overlay
for sel_thresh in sel_threshs:
    nrows = len(components) // 4 + 1
    fig, axs = plt.subplots(ncols=4, nrows=nrows, figsize=(20, 5 * nrows))

    for i, k in enumerate(components):
        ax = axs.flat[i]

        real_pvals = test_stats_dfs.loc[
            (test_stats_dfs.K == k) & (test_stats_dfs.real == True) & (test_stats_dfs.sel_thresh == sel_thresh), 'pval']
        null_pvals = test_stats_dfs.loc[
            (test_stats_dfs.K == k) & (test_stats_dfs.real == False) &
            (test_stats_dfs.target_name == 'targeting') & (test_stats_dfs.sel_thresh == sel_thresh), 'pval']

        if len(real_pvals) > 0:
            qqplot(data=real_pvals, ax=ax, color='blue', label='Real')
        if len(null_pvals) > 0:
            qqplot(data=null_pvals, ax=ax, color='red', label='Null')

        ax.set_xlim(ax.get_xlim()[0], 6)
        ax.set_title(f'K={k}')
        ax.legend()

    for j in range(i + 1, len(axs.flat)):
        axs.flat[j].axis('off')
    plt.tight_layout()

    thresh_str = str(sel_thresh).replace('.', '_')
    plt.savefig(f'{plot_dir}/perturbation_association_qqplot_overlap_{thresh_str}.png', dpi=100)
    plt.show()